In [ ]:
import torch
import torch.nn as nn

class AttentionCompressor(nn.Module):
    def __init__(self, input_len, target_len=77, embed_dim=1024, heads=8):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, target_len, embed_dim))
        self.attn = nn.MultiheadAttention(embed_dim, heads, batch_first=True)
        
    def forward(self, x):
        q = self.query.expand(x.size(0), -1, -1)
        out, _ = self.attn(q, x, x)
        return out

class TokenCompressor(nn.Module):
    def __init__(self, input_len, vocab_size , target_len=77, embed_dim=1024):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(input_len, embed_dim)
        self.proj = nn.Linear(input_len, target_len, bias=False)
        self.norm = nn.LayerNorm(embed_dim)
        self.register_buffer(
            "position_ids", torch.arange(input_len).expand((1, -1)), persistent=False
        )

    def forward(self, input_ids):
        seq_length = input_ids.shape[-1] if input_ids is not None else inputs_embeds.shape[-2]
        position_ids = self.position_ids[:, :seq_length]
        inputs_embeds = self.token_embedding(input_ids)

        position_embeddings = self.position_embedding(position_ids)
        embeddings = inputs_embeds + position_embeddings
        # x shape: (batch, input_len, embed_dim)
        input_ids = embeddings.transpose(1, 2)  # (batch, embed_dim, input_len)
        input_ids = self.proj(input_ids)       # (batch, embed_dim, target_len)
        input_ids = input_ids.transpose(1, 2)  # (batch, target_len, embed_dim)
        return self.norm(input_ids)


In [ ]:
class EarlyStop:
    def __init__(self, max_iter):
        self.max_iteration = max_iter
        self.best_loss = None
        self.counter = 0
        self.need_stop = False
        self.loss_history=[]

    def step(self, val_loss):
        self.loss_history.append(val_loss)
        if self.best_loss is None:
            self.best_loss = val_loss
        elif self.best_loss > val_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.max_iteration:
                self.need_stop = True
        

In [ ]:
device = 'cuda'
token_vocab_size = 49408 #CLIPTextModel
batch_size = 32
epoches = 400

In [ ]:
from transformers import CLIPTextModel, CLIPTokenizer
# from transformers import CLIPTextModel, CLIPTokenizer
# clip_model = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").eval().to(device)
# clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")


base_model_name = 'stabilityai/sd-turbo'

tokenizer = CLIPTokenizer.from_pretrained(
    base_model_name,
    subfolder="tokenizer",
)

text_encoder = CLIPTextModel.from_pretrained(
    base_model_name, subfolder="text_encoder"
)

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd

class CompressorDataset(Dataset):
    def __init__(self, data_file):
        super().__init__()
        self.data = pd.read_csv(data_file)
        self.len = self.data.shape[0]
    
    def __len__(self):
        return self.len

    def __getitem__(self, index):
        if index >= self.len:
            raise 'Index out of range'
        
        item = self.data.iloc[index]
        return  item['long_text'], item['short_text']#{'long_text':item[0],'short_text':item[1]}

In [ ]:
compressor_ds = CompressorDataset(r'/kaggle/input/text-compressor-ds/text_compressor_ds.csv')

generator = torch.Generator().manual_seed(27)
train_set_size = int(0.8* len(compressor_ds))
train_dataset, val_dataset = random_split(compressor_ds, [train_set_size, len(compressor_ds)-train_set_size], generator=generator)

train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True)
validation_dataloader = DataLoader(val_dataset, batch_size, shuffle=True)

In [ ]:
from tqdm.auto import tqdm

comp = TokenCompressor(300, token_vocab_size, 77)
optimizer = torch.optim.AdamW(comp.parameters(), lr=5e-4)
loss_history = []
text_encoder.to(device).eval()
text_encoder.requires_grad_(False)

early_stoper = EarlyStop(10000)

progress_bar = tqdm(range(epoches * len(train_dataloader)), disable=False)
progress_bar.set_description("Steps")

comp.to(device)

comp.train()
for e in range(epoches):
    for batch in train_dataloader:
        long_texts, short_texts = batch
        long_ids = tokenizer(long_texts, return_tensors='pt', padding='max_length', truncation=True, max_length=300).input_ids.to(device)
        short_ids = tokenizer(short_texts, return_tensors='pt', padding='max_length', truncation=True, max_length=77).input_ids.to(device)

        with torch.no_grad():
            teacher_emb = text_encoder(short_ids)[0]

        student_emb = comp(long_ids)
        loss = torch.nn.functional.mse_loss(student_emb, teacher_emb)
        logs = {"loss": loss.detach().item()}
        loss_history.append(logs['loss'])
        early_stoper.step(logs['loss'])
        if early_stoper.need_stop:
            print('Early Stopping is triggered.')
            break
        progress_bar.update(1)
        progress_bar.set_postfix(**logs)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [ ]:
torch.save(comp.state_dict(),r'compressor_weight_400E.pt')

In [ ]:
import numpy as np

comp.eval()        
val_loss = []
with torch.no_grad():
    for batch in validation_dataloader:
        long_texts, short_texts = batch
        long_ids = tokenizer(long_texts, return_tensors='pt', padding='max_length', truncation=True, max_length=300).input_ids.to(device)
        short_ids = tokenizer(short_texts, return_tensors='pt', padding='max_length', truncation=True, max_length=77).input_ids.to(device)
        teacher_emb = text_encoder(short_ids)[0]
        student_emb = comp(long_ids)
        loss = torch.nn.functional.mse_loss(student_emb, teacher_emb)
        val_loss.append(loss.detach().item())
        


validation_val = np.mean(val_loss)
print('Validation Loss:', validation_val)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(loss_history, label="Training Loss", color="blue", linewidth=2)
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()
plt.grid(True)

plt.savefig('loss_curver.png', dpi=600)

plt.show()